In [9]:
import os
import pandas as pd
import plotly.graph_objects as go
from typing import List, Optional, Dict
import numpy as np

def find_config_file(folder_path: str) -> Optional[str]:
    """
    Finds a configuration file (ending with .txt) within the 'configs' subfolder.
    """
    config_dir = os.path.join(folder_path, 'configs')
    if not os.path.isdir(config_dir):
        return None
    for item in os.listdir(config_dir):
        if item.endswith('.txt'):
            return os.path.join(config_dir, item)
    return None

def parse_config(file_path: str) -> Dict[str, str]:
    """
    Parses a 'key = value', 'key value', or 'key: value' configuration file into a dictionary.
    """
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                separator = None
                if ':' in line:
                    separator = ':'
                elif '=' in line:
                    separator = '='

                if separator:
                    parts = line.split(separator, 1)
                else:
                    parts = line.split(None, 1)

                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except FileNotFoundError:
        print(f"Config file not found: {file_path}")
    except Exception as e:
        print(f"Error parsing config file {file_path}: {e}")
    return params

def find_timing_file(run_folder_path: str, sim_type: str) -> Optional[str]:
    """Finds the ...trace_matched_timing.csv file for a given run."""
    sim_output_dir = os.path.join(run_folder_path, sim_type.lower())
    if not os.path.isdir(sim_output_dir):
        return None
    for f in os.listdir(sim_output_dir):
        if 'trace_matched_timing.csv' in f:
            return os.path.join(sim_output_dir, f)
    return None

# --- Collective Info Parsing ---
def parse_collectives_log(log_path: str) -> Dict[str, Dict]:
    """Parses a duplicate_collectives.log file to extract signatures."""
    collective_info = {}
    try:
        with open(log_path, 'r') as f:
            lines = f.readlines()
            i = 0
            while i < len(lines):
                line = lines[i]
                workload_match = re.match(r'^Workload: (\S+)', line)
                if workload_match:
                    current_workload = workload_match.group(1)
                    # Look for signature on the next line
                    if (i + 1 < len(lines)) and (signature_match := re.match(r'^\s+Signature: \((.*)\)', lines[i+1])):
                        sig_content = signature_match.group(1).strip()
                        # Split signature into its three parts
                        parts = sig_content.rsplit(', ', 2)
                        if len(parts) == 3:
                            npu_tuples, comm_type, comm_size = parts
                            collective_info[current_workload] = {
                                'npu_tuples': npu_tuples.strip(),
                                'comm_type': comm_type.strip(),
                                'comm_size': comm_size.strip()
                            }
                i += 1
    except FileNotFoundError:
        print(f"Warning: Collectives log file not found at {log_path}")
    except Exception as e:
        print(f"Error parsing collectives log {log_path}: {e}")
    return collective_info

In [11]:
import re
import pandas as pd
from plotly.subplots import make_subplots
# --- Configuration for G2 vs NS3 Comparison ---

# Define the base folder containing all the run directories from all workloads
# We will process all workloads ('toy_all_to_all_one_collective', 'toy_all_reduce_one_collective', 'model')
# base_comparison_folder = '/app/astra-sim/upc/output/comparison_run/FoldedClos'
# base_comparison_folder = '/app/astra-sim/upc/output/comparison_run/FoldedClos/basic_model_0_split'
base_comparison_folders = [
    '/app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Base_split',
    '/app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Base_split_2',
    '/app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Base_split_3',
    '/app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Base_split_4']

# Choose what to plot: 'avg' for mean, or 'max' for maximum value
comparison_plot_metric = 'avg'  # Can be 'avg' or 'max'

# --- Data Collection Logic ---

all_run_folders = []
for base_comparison_folder in base_comparison_folders:
    for workload_folder in os.listdir(base_comparison_folder):
        workload_path = os.path.join(base_comparison_folder, workload_folder)
        if os.path.isdir(workload_path):
            run_folders = [os.path.join(workload_path, d) for d in os.listdir(workload_path) if os.path.isdir(os.path.join(workload_path, d))]
            all_run_folders.extend(run_folders)

comparison_results = []

# Regex to extract topology index
topo_idx_regex = re.compile(r'topology(\d+)(?:\.json)?$')

cc_modes = {0: "PFC", 1: "DCQCN", 3: "HPCC", 7: "TIMELY", 8: "DCTCP", 10: "HPCC-PINT"}

def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into seconds."""
    if not time_str:
        return 0.0
    try:
        parts = time_str.split(':')
        h = int(parts[0])
        m = int(parts[1])
        s = float(parts[2])
        return h * 3600 + m * 60 + s
    except (ValueError, IndexError):
        return 0.0

if base_comparison_folders:
    # e.g., /app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Base_split -> /app/astra-sim/upc/comparing_networks/workload/T5_Base_split/duplicate_collectives.log
    base_name = os.path.basename(base_comparison_folders[0])
    log_file_path = f'/app/astra-sim/upc/comparing_networks/workload/{base_name}/duplicate_collectives.log'

collectives_data = parse_collectives_log(log_file_path)
if not collectives_data:
    print("Could not load collective signature data. Columns will be empty.")


for folder in sorted(all_run_folders):
    run_summary_path = os.path.join(folder, 'run_summary.txt')
    if not os.path.exists(run_summary_path):
        continue

    # 1. Parse run_summary.txt to identify sim_type and topology
    summary_params = parse_config(run_summary_path)
    
    workload_name = summary_params.get('collective', 'N/A').strip()
    npu_count = summary_params.get('npus count', 'N/A')
    total_runtime_str = summary_params.get('total runtime', '0:0:0.0')
    execution_time_sec = parse_runtime(total_runtime_str)

    # Get collective info
    collective_info = collectives_data.get(os.path.basename(workload_name), {})
    npu_tuples = collective_info.get('npu_tuples', 'N/A')
    comm_type = collective_info.get('comm_type', 'N/A')
    comm_size = collective_info.get('comm_size', 'N/A')

    # Process 'analytical_unaware' separately as it's topology-independent
    if os.path.exists(os.path.join(folder, 'analytical_unaware')):
        timing_file = None
        sim_output_dir = os.path.join(folder, 'analytical_unaware')
        for f in os.listdir(sim_output_dir):
            if 'trace_matched_timing.csv' in f:
                timing_file = os.path.join(sim_output_dir, f)
                break
        
        if timing_file:
            try:
                df = pd.read_csv(timing_file)
                time_col = 'callback_tick'
                if time_col in df.columns:
                    elapsed_times = df[time_col].dropna()
                    if not elapsed_times.empty:
                        comparison_results.append({
                            'workload': workload_name,
                            'npu_count': npu_count,
                            'npu_tuples': npu_tuples,
                            'comm_type': comm_type,
                            'comm_size': comm_size,
                            'topo_index': -1,  # Use -1 to indicate topology independence
                            'sim_type': 'Analytical Unaware',
                            'run_name': 'Analytical Unaware',
                            'avg_time': elapsed_times.mean(),
                            'max_time': elapsed_times.max(),
                            'std_dev': elapsed_times.std(),
                            'execution_time': execution_time_sec,
                            'path': folder
                        })
            except Exception as e:
                print(f"Error processing analytical_unaware in {folder}: {e}")

    # Process topology-dependent simulations (G2, NS3)
    sim_type = None
    topo_file = None
    if os.path.exists(os.path.join(folder, 'g2')):
        sim_type = 'G2'
        topo_file = summary_params.get('g2 topology file override', 'N/A')
    elif os.path.exists(os.path.join(folder, 'ns3')):
        sim_type = 'NS3'
        topo_file = summary_params.get('ns3 topology file override', 'N/A')

    if not sim_type or not topo_file or 'all_paths' in topo_file:
        continue

    # Extract topology index
    match = topo_idx_regex.search(topo_file)
    if not match:
        continue
    topo_index = int(match.group(1))

    # 2. Get timing data
    timing_file = None
    sim_output_dir = None
    if sim_type == 'G2':
        sim_output_dir = os.path.join(folder, 'g2')
    elif sim_type == 'NS3':
        sim_output_dir = os.path.join(folder, 'ns3')

    if sim_output_dir and os.path.isdir(sim_output_dir):
        for f in os.listdir(sim_output_dir):
            if 'trace_matched_timing.csv' in f:
                timing_file = os.path.join(sim_output_dir, f)
                break
    
    if not timing_file:
        continue

    try:
        df = pd.read_csv(timing_file)
        if sim_type == 'NS3':
            df = df[df['node_name'] != 'dummy_node'].copy()
        
        time_col = 'callback_tick'
        if time_col not in df.columns:
            continue
            
        elapsed_times = df[time_col].dropna()
        if elapsed_times.empty:
            continue

        # 3. Create a descriptive name and store results
        run_name = f"{sim_type}"
        if sim_type == 'NS3':
            ns3_config_file = find_config_file(folder)
            if ns3_config_file:
                ns3_params = parse_config(ns3_config_file)
                run_name = (
                    f"NS3 (cc:{cc_modes.get(int(ns3_params.get('cc_mode', 'N/A')), 'N/A')}, "
                    f"win:{ns3_params.get('has_win', 'N/A')}, "
                    f"adapt:{ns3_params.get('var_win', 'N/A')}, "
                    f"buf:{ns3_params.get('buffer_size', 'N/A')}, "
                    f"size:{ns3_params.get('packet_payload_size', 'N/A')})"
                )

        comparison_results.append({
            'workload': workload_name,
            'npu_count': npu_count,
            'npu_tuples': npu_tuples,
            'comm_type': comm_type,
            'comm_size': comm_size,
            'topo_index': topo_index,
            'sim_type': sim_type,
            'run_name': run_name,
            'avg_time': elapsed_times.mean(),
            'max_time': elapsed_times.max(),
            'min_time': elapsed_times.min(),
            'std_dev': elapsed_times.std(),
            'execution_time': execution_time_sec,
            'path': folder
        })

    except Exception as e:
        print(f"Error processing {folder}: {e}")

# --- Plotting Logic ---

if comparison_results:
    comp_df = pd.DataFrame(comparison_results)
    
    # Determine which column to use for plotting
    if comparison_plot_metric == 'max':
        y_col = 'max_time'
        y_axis_title = "Maximum Time (ns)"
    elif comparison_plot_metric == 'min':
        y_col = 'min_time'
        y_axis_title = "Minimum Time (ns)"
    else: # Default to 'avg'
        y_col = 'avg_time'
        y_axis_title = "Average Time (ns)"

    workloads = comp_df['workload'].unique()
    topology_name = os.path.basename(base_comparison_folder)
    
    for wl in sorted(workloads):
        workload_df = comp_df[comp_df['workload'] == wl]
        
        # For this workload, find the single analytical unaware time, if it exists.
        au_runs = workload_df[workload_df['sim_type'] == 'Analytical Unaware']
        au_time = None
        if not au_runs.empty:
            au_time = au_runs.iloc[0][y_col]

        for topo_idx in sorted(workload_df['topo_index'].unique()):
            # Skip the placeholder index used for analytical_unaware
            if topo_idx == -1:
                continue

            group_df = workload_df[workload_df['topo_index'] == topo_idx].copy()
            
            if group_df.empty:
                continue

            # Separate G2 and NS3 for plotting
            g2_runs = group_df[group_df['sim_type'] == 'G2']
            ns3_runs = group_df[group_df['sim_type'] == 'NS3'].sort_values(by=y_col)
            
            if ns3_runs.empty:
                continue # Don't plot if there's no NS3 data to compare against

            plot_title = f'G2 vs NS3 Comparison for Workload: "{wl}", Static routing: {topo_idx}'

            fig = make_subplots(
                rows=1, cols=2,
                subplot_titles=("Performance Comparison", "Per-NPU Time Correlation"),
                column_widths=[0.6, 0.4],
                specs=[[{"type": "xy"}, {"type": "xy"}]]
            )

            # Add NS3 runs as bars to the first subplot
            fig.add_trace(go.Bar(
                x=ns3_runs['run_name'],
                y=ns3_runs[y_col],
                name='NS3 Runs',
                marker_color='rgb(55, 83, 109)',
                text=ns3_runs[y_col].apply(lambda x: f'{x/1e9:.4f} s'),
                textposition='outside'
            ), row=1, col=1)

            # --- Speedup Calculation & Annotation ---
            fastest_ns3_run = ns3_runs.iloc[0]
            fastest_ns3_time = fastest_ns3_run[y_col]
            npu_count_val = group_df['npu_count'].iloc[0] if not group_df.empty else 'N/A'
            
            g2_time = None
            g2_sim_time_error_text = "G2: N/A (No G2 run)"
            if not g2_runs.empty:
                g2_run = g2_runs.iloc[0]
                g2_time = g2_run[y_col]
                # Calculate percentage error relative to fastest NS3
                sim_time_error_pct = ((g2_time - fastest_ns3_time) / fastest_ns3_time) * 100
                g2_sim_time_error_text = f"G2: {sim_time_error_pct:+.2f}%"
                fig.add_hline(
                    y=g2_time, 
                    line_dash="dot",
                    annotation_text=f"G2 Time: {g2_time/1e9:.4f} s", 
                    annotation_position="top right",
                    line_color="red",
                    annotation=dict(font=dict(color="white", size=12), bgcolor="red", borderpad=4),
                    row=1, col=1
                )

            au_sim_time_error_text = "Unaware: N/A"
            if au_time is not None:
                # Calculate percentage error relative to fastest NS3
                sim_time_error_pct = ((au_time - fastest_ns3_time) / fastest_ns3_time) * 100
                au_sim_time_error_text = f"Unaware: {sim_time_error_pct:+.2f}%"
                fig.add_hline(
                    y=au_time, 
                    line_dash="dash",
                    annotation_text=f"Analytical Unaware: {au_time/1e9:.4f} s", 
                    annotation_position="bottom right",
                    line_color="green",
                    annotation=dict(font=dict(color="white", size=12), bgcolor="green", borderpad=4),
                    row=1, col=1
                )

            # --- Scatter plot for the second subplot ---
            if not g2_runs.empty:
                g2_timing_file = find_timing_file(g2_run['path'], 'G2')
                ns3_timing_file = find_timing_file(fastest_ns3_run['path'], 'NS3')

                if g2_timing_file and ns3_timing_file:
                    try:
                        df_g2 = pd.read_csv(g2_timing_file)
                        df_ns3 = pd.read_csv(ns3_timing_file)

                        ns3_times = df_ns3[['sys_id', 'callback_tick']].rename(columns={'callback_tick': 'ns3_time'})
                        g2_times = df_g2[['sys_id', 'callback_tick']].rename(columns={'callback_tick': 'g2_time'})
                        merged_df = pd.merge(ns3_times, g2_times, on='sys_id')

                        merged_df = merged_df[(merged_df['ns3_time'] >= 100) & (merged_df['g2_time'] >= 100)]

                        fig.add_trace(go.Scatter(
                            x=merged_df['ns3_time'],
                            y=merged_df['g2_time'],
                            mode='markers', name='NPU Finish Times',
                            marker=dict(color='blue'),
                            text=merged_df['sys_id'].apply(lambda x: f'NPU {x}'),
                            hoverinfo='text+x+y'
                        ), row=1, col=2)

                        min_val = min(merged_df['ns3_time'].min(), merged_df['g2_time'].min())
                        max_val = max(merged_df['ns3_time'].max(), merged_df['g2_time'].max())
                        fig.add_trace(go.Scatter(
                            x=[min_val, max_val], y=[min_val, max_val],
                            mode='lines', name='Ideal Match (y=x)',
                            line=dict(color='red', dash='dash')
                        ), row=1, col=2)
                    except Exception as e:
                        print(f"Could not generate scatter plot for {wl} topo {topo_idx}: {e}")

            # --- Execution Time Speedup Calculation ---
            fastest_ns3_exec_time = ns3_runs['execution_time'].min()

            g2_exec_speedup_text = "N/A"
            if not g2_runs.empty:
                g2_exec_time = g2_runs.iloc[0]['execution_time']
                if g2_exec_time > 0:
                    exec_speedup = fastest_ns3_exec_time / g2_exec_time
                    g2_exec_speedup_text = f"{exec_speedup:.2f}x"

            au_exec_speedup_text = "N/A"
            if not au_runs.empty:
                au_exec_time = au_runs.iloc[0]['execution_time']
                if au_exec_time > 0:
                    exec_speedup = fastest_ns3_exec_time / au_exec_time
                    au_exec_speedup_text = f"{exec_speedup:.2f}x"

            # Construct the summary text
            summary_text = (
                f"<b>Summary</b><br>"
                f"--------------------<br>"
                f"<b>Topology:</b> {topology_name}<br>"
                f"<b>NPU Nodes:</b> {npu_count_val}<br>"
                f"<b>Static Routing:</b> {topo_idx}<br>"
                f"--------------------<br>"
                f"<b>Execution Time Error vs Fastest NS3 (%):</b><br>"
                f"- {g2_sim_time_error_text}<br>"
                f"- {au_sim_time_error_text}<br>"
                f"--------------------<br>"
                f"<b>Sim Time Speedup vs Fastest NS3:</b><br>"
                f"- G2: {g2_exec_speedup_text}<br>"
                # f"- Unaware: {au_exec_speecdup_text}"
            )

            fig.add_annotation(
                text=summary_text,
                align='left',
                showarrow=False,
                xref='paper',
                yref='paper',
                x=1.29,
                y=0.8,
                bordercolor="black",
                borderwidth=1,
                bgcolor="rgba(255, 255, 255, 0.8)"
            )

            fig.update_layout(
                title=plot_title,
                template='plotly_white',
                height=700,
                width=1600,
                margin=dict(b=350, r=450), # Increased right margin for the text box
                legend=dict(x=1.05, y=1.0),
                showlegend=True
            )
            
            # Update axes for both subplots
            fig.update_xaxes(title_text="Run Configuration", tickangle=-60, row=1, col=1)
            fig.update_yaxes(title_text=y_axis_title, row=1, col=1)
            
            fig.update_xaxes(title_text=f"Fastest NS3 Time (ns)", row=1, col=2)
            fig.update_yaxes(title_text="G2 Time (ns)", row=1, col=2)

            # fig.show()
else:
    print("\nNo comparison results to plot.")

# Display the full data table
if comparison_results:
    print("\n--- Full Comparison Data ---")
    # Reorder columns for better readability
    display_cols = [
        'workload', 'topo_index', 'sim_type', 'run_name', 
        y_col, 'std_dev', 'execution_time', 'npu_count', 
        'npu_tuples', 'comm_type', 'comm_size', 'path'
    ]
    # Ensure all columns exist before trying to display them
    final_cols = [c for c in display_cols if c in comp_df.columns]
    with pd.option_context('display.max_rows', 10, 'display.max_columns', None, 'display.width', 1000):
        display(comp_df.sort_values(by=['workload', 'topo_index', y_col])[final_cols])



--- Full Comparison Data ---


,workload,topo_index,sim_type,run_name,avg_time,std_dev,execution_time,npu_count,npu_tuples,comm_type,comm_size,path
0,T5_Base_split/0000_T5_Base_multiple_2_2_2_2_1_...,-1,Analytical Unaware,Analytical Unaware,3.725290e+09,3.847463e+09,0.934860,16,"((0, 2), (1, 3), (4, 6), (5, 7))",2,16777216,/app/astra-sim/upc/output/comparison_run/Folde...
1,T5_Base_split/0000_T5_Base_multiple_2_2_2_2_1_...,1,G2,G2,7.555343e+09,8.267075e+09,0.631840,16,"((0, 2), (1, 3), (4, 6), (5, 7))",2,16777216,/app/astra-sim/upc/output/comparison_run/Folde...
2,T5_Base_split/0000_T5_Base_multiple_2_2_2_2_1_...,1,NS3,"NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500)",7.600936e+09,8.285868e+09,10.196461,16,"((0, 2), (1, 3), (4, 6), (5, 7))",2,16777216,/app/astra-sim/upc/output/comparison_run/Folde...
3,T5_Base_split/0000_T5_Base_multiple_2_2_2_2_1_...,1,NS3,"NS3 (cc:DCQCN, win:1, adapt:1, buf:1, size:1500)",8.859424e+09,9.149974e+09,22.902080,16,"((0, 2), (1, 3), (4, 6), (5, 7))",2,16777216,/app/astra-sim/upc/output/comparison_run/Folde...
4,T5_Base_split/0000_T5_Base_multiple_2_2_2_2_1_...,1,NS3,"NS3 (cc:PFC, win:1, adapt:1, buf:1, size:1500)",8.859424e+09,9.149974e+09,10.042329,16,"((0, 2), (1, 3), (4, 6), (5, 7))",2,16777216,/app/astra-sim/upc/output/comparison_run/Folde...
...,...,...,...,...,...,...,...,...,...,...,...,...
2143,T5_Base_split_4/0109_T5_Base_multiple_8_2_1_1_...,-1,Analytical Unaware,Analytical Unaware,4.889449e+09,0.000000e+00,0.995093,16,"((0, 1, 2, 3, 4, 5, 6, 7), (8, 9, 10, 11, 12, ...",0,6291456,/app/astra-sim/upc/output/comparison_run/Folde...
2144,T5_Base_split_4/0109_T5_Base_multiple_8_2_1_1_...,4,G2,G2,5.666729e+09,0.000000e+00,0.701001,16,"((0, 1, 2, 3, 4, 5, 6, 7), (8, 9, 10, 11, 12, ...",0,6291456,/app/astra-sim/upc/output/comparison_run/Folde...
2145,T5_Base_split_4/0109_T5_Base_multiple_8_2_1_1_...,4,NS3,"NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500)",6.870163e+09,9.076013e+07,12.048677,16,"((0, 1, 2, 3, 4, 5, 6, 7), (8, 9, 10, 11, 12, ...",0,6291456,/app/astra-sim/upc/output/comparison_run/Folde...
2146,T5_Base_split_4/0109_T5_Base_multiple_8_2_1_1_...,4,NS3,"NS3 (cc:DCQCN, win:1, adapt:1, buf:1, size:1500)",6.901230e+09,1.129925e+08,21.979729,16,"((0, 1, 2, 3, 4, 5, 6, 7), (8, 9, 10, 11, 12, ...",0,6291456,/app/astra-sim/upc/output/comparison_run/Folde...


In [22]:
if comparison_results:
    comp_df = pd.DataFrame(comparison_results)
    
    # Ensure min_time and max_time columns exist, fill with NaN if not
    if 'min_time' not in comp_df.columns:
        comp_df['min_time'] = pd.NA
    if 'max_time' not in comp_df.columns:
        comp_df['max_time'] = pd.NA

    def analyze_and_display_divergence(comp_df, metric_col, metric_name):
        """
        Analyzes and displays the divergence between G2 and NS3 DCTCP for a given metric.
        """
        divergence_data = []

        for wl, group in comp_df.groupby('workload'):
            g2_runs = group[group['sim_type'] == 'G2']
            ns3_dctcp_runs = group[(group['sim_type'] == 'NS3') & (group['run_name'].str.contains('DCTCP'))]

            if g2_runs.empty or ns3_dctcp_runs.empty:
                continue

            g2_metric = g2_runs[metric_col].mean()
            ns3_metric = ns3_dctcp_runs[metric_col].mean()

            if pd.notna(g2_metric) and pd.notna(ns3_metric) and ns3_metric > 0:
                # Get collective info from the first row of the group
                info = group.iloc[0]
                
                divergence_data.append({
                    'Workload': wl,
                    'comm_type': info['comm_type'],
                    'comm_size': info['comm_size'],
                    'npu_tuples': info['npu_tuples'],
                    f'G2 {metric_name} (ns)': g2_metric,
                    f'NS3 DCTCP {metric_name} (ns)': ns3_metric,
                    'Abs Diff (ns)': g2_metric - ns3_metric,
                    'Rel Diff (%)': ((g2_metric - ns3_metric) / ns3_metric) * 100
                })

        if divergence_data:
            print(f"--- Analysis: Top 20 Divergences in {metric_name} (G2 vs DCTCP, Averaged over Topologies) ---")
            div_df = pd.DataFrame(divergence_data)
            div_df['abs_rel_diff'] = div_df['Rel Diff (%)'].abs()
            display(div_df.sort_values(by='abs_rel_diff', ascending=False).drop(columns='abs_rel_diff').head(20))
        else:
            print(f"No data to analyze {metric_name.lower()} divergence for G2 vs DCTCP.")

    # --- Displaying Divergence Tables ---
    pd.set_option('display.float_format', '{:,.2f}'.format)
    
    analyze_and_display_divergence(comp_df, 'avg_time', 'Average NPU Finish Time')
    print("\n")
    analyze_and_display_divergence(comp_df, 'min_time', 'Minimum NPU Finish Time')
    print("\n")
    analyze_and_display_divergence(comp_df, 'max_time', 'Maximum NPU Finish Time')

else:
    print("No comparison results available to perform divergence analysis.")

--- Analysis: Top 20 Divergences in Average NPU Finish Time (G2 vs DCTCP, Averaged over Topologies) ---


,Workload,comm_type,comm_size,npu_tuples,G2 Average NPU Finish Time (ns),NS3 DCTCP Average NPU Finish Time (ns),Abs Diff (ns),Rel Diff (%)
369,T5_Base_split_4/0047_T5_Base_multiple_4_2_2_1_...,2,8388608,"((0, 4), (1, 5), (2, 6), (3, 7), (8, 12), (9, ...","6,206,190,645.00","8,526,141,619.69","-2,319,950,974.69",-27.21
370,T5_Base_split_4/0048_T5_Base_multiple_4_2_2_1_...,7,16777216,"((0, 4), (1, 5), (2, 6), (3, 7), (8, 12), (9, ...","6,206,198,157.00","8,526,149,131.69","-2,319,950,974.69",-27.21
334,T5_Base_split_4/0010_T5_Base_multiple_2_2_2_2_...,2,16777216,"((8, 12), (9, 13), (10, 14), (11, 15))","6,476,008,524.50","8,885,255,297.62","-2,409,246,773.12",-27.12
333,T5_Base_split_4/0009_T5_Base_multiple_2_2_2_2_...,7,33554432,"((8, 12), (9, 13), (10, 14), (11, 15))","6,476,016,036.50","8,885,262,809.62","-2,409,246,773.12",-27.12
327,T5_Base_split_4/0003_T5_Base_multiple_2_2_2_2_...,2,16777216,"((0, 4), (1, 5), (2, 6), (3, 7))","5,936,341,149.50","8,111,887,782.81","-2,175,546,633.31",-26.82
326,T5_Base_split_4/0002_T5_Base_multiple_2_2_2_2_...,7,33554432,"((0, 4), (1, 5), (2, 6), (3, 7))","5,936,348,661.50","8,111,895,294.81","-2,175,546,633.31",-26.82
264,T5_Base_split_3/0050_T5_Base_multiple_4_2_2_1_...,2,8388608,"((0, 8), (1, 9), (2, 10), (3, 11), (4, 12), (5...","6,476,025,262.81","8,836,491,729.00","-2,360,466,466.19",-26.71
263,T5_Base_split_3/0049_T5_Base_multiple_4_2_2_1_...,7,16777216,"((0, 8), (1, 9), (2, 10), (3, 11), (4, 12), (5...","6,476,032,774.81","8,836,499,241.00","-2,360,466,466.19",-26.71
396,T5_Base_split_4/0074_T5_Base_multiple_1_4_2_2_...,0,3145728,"((8, 12), (9, 13), (10, 14), (11, 15))","1,315,453,996.75","1,738,243,555.38","-422,789,558.62",-24.32
336,T5_Base_split_4/0012_T5_Base_multiple_2_2_2_2_...,0,6291456,"((8, 12), (9, 13), (10, 14), (11, 15))","2,630,907,968.75","3,448,858,554.69","-817,950,585.94",-23.72




--- Analysis: Top 20 Divergences in Minimum NPU Finish Time (G2 vs DCTCP, Averaged over Topologies) ---


,Workload,comm_type,comm_size,npu_tuples,G2 Minimum NPU Finish Time (ns),NS3 DCTCP Minimum NPU Finish Time (ns),Abs Diff (ns),Rel Diff (%)
297,T5_Base_split_3/0083_T5_Base_multiple_1_8_2_1_...,0,1572864,"((0, 8), (1, 9), (2, 10), (3, 11), (4, 12), (5...","809,532,732.00","1,485,636,732.00","-676,104,000.00",-45.51
405,T5_Base_split_4/0083_T5_Base_multiple_1_8_2_1_...,0,1572864,"((0, 8), (1, 9), (2, 10), (3, 11), (4, 12), (5...","809,532,732.00","1,485,636,732.00","-676,104,000.00",-45.51
311,T5_Base_split_3/0097_T5_Base_multiple_2_4_2_1_...,0,3145728,"((0, 8), (1, 9), (2, 10), (3, 11), (4, 12), (5...","1,619,021,437.00","2,963,269,437.00","-1,344,248,000.00",-45.36
419,T5_Base_split_4/0097_T5_Base_multiple_2_4_2_1_...,0,3145728,"((0, 8), (1, 9), (2, 10), (3, 11), (4, 12), (5...","1,619,021,437.00","2,963,269,437.00","-1,344,248,000.00",-45.36
374,T5_Base_split_4/0052_T5_Base_multiple_4_2_2_1_...,0,6291456,"((0, 8), (1, 9), (2, 10), (3, 11), (4, 12), (5...","3,238,042,847.00","5,926,298,847.00","-2,688,256,000.00",-45.36
266,T5_Base_split_3/0052_T5_Base_multiple_4_2_2_1_...,0,6291456,"((0, 8), (1, 9), (2, 10), (3, 11), (4, 12), (5...","3,238,042,847.00","5,926,298,847.00","-2,688,256,000.00",-45.36
267,T5_Base_split_3/0053_T5_Base_multiple_4_2_2_1_...,0,8388608,"((0, 8), (1, 9), (2, 10), (3, 11), (4, 12), (5...","4,317,375,786.00","7,898,235,786.00","-3,580,860,000.00",-45.34
375,T5_Base_split_4/0053_T5_Base_multiple_4_2_2_1_...,0,8388608,"((0, 8), (1, 9), (2, 10), (3, 11), (4, 12), (5...","4,317,375,786.00","7,898,235,786.00","-3,580,860,000.00",-45.34
282,T5_Base_split_3/0068_T5_Base_multiple_8_1_2_1_...,0,12582912,"((0, 8), (1, 9), (2, 10), (3, 11), (4, 12), (5...","6,476,041,664.00","11,847,101,664.00","-5,371,060,000.00",-45.34
390,T5_Base_split_4/0068_T5_Base_multiple_8_1_2_1_...,0,12582912,"((0, 8), (1, 9), (2, 10), (3, 11), (4, 12), (5...","6,476,041,664.00","11,847,101,664.00","-5,371,060,000.00",-45.34




--- Analysis: Top 20 Divergences in Maximum NPU Finish Time (G2 vs DCTCP, Averaged over Topologies) ---


,Workload,comm_type,comm_size,npu_tuples,G2 Maximum NPU Finish Time (ns),NS3 DCTCP Maximum NPU Finish Time (ns),Abs Diff (ns),Rel Diff (%)
219,T5_Base_split_3/0003_T5_Base_multiple_2_2_2_2_...,2,16777216,"((0, 4), (1, 5), (2, 6), (3, 7))","17,269,356,020.00","24,107,393,054.00","-6,838,037,034.00",-28.36
218,T5_Base_split_3/0002_T5_Base_multiple_2_2_2_2_...,7,33554432,"((0, 4), (1, 5), (2, 6), (3, 7))","17,269,371,044.00","24,107,408,078.00","-6,838,037,034.00",-28.36
396,T5_Base_split_4/0074_T5_Base_multiple_1_4_2_2_...,0,3145728,"((8, 12), (9, 13), (10, 14), (11, 15))","3,238,040,140.00","4,354,332,438.00","-1,116,292,298.00",-25.64
330,T5_Base_split_4/0006_T5_Base_multiple_2_2_2_2_...,0,8388608,"((0, 4), (1, 5), (2, 6), (3, 7))","8,634,744,020.00","11,416,473,528.00","-2,781,729,508.00",-24.37
329,T5_Base_split_4/0005_T5_Base_multiple_2_2_2_2_...,0,6291456,"((0, 4), (1, 5), (2, 6), (3, 7))","6,476,080,020.00","8,528,675,788.00","-2,052,595,768.00",-24.07
337,T5_Base_split_4/0013_T5_Base_multiple_2_2_2_2_...,0,8388608,"((8, 12), (9, 13), (10, 14), (11, 15))","8,634,744,343.00","11,347,034,331.00","-2,712,289,988.00",-23.90
189,T5_Base_split_2/0083_T5_Base_multiple_1_8_2_1_...,0,1572864,"((0, 8), (1, 9), (2, 10), (3, 11), (4, 12), (5...","2,428,596,248.00","3,184,605,736.00","-756,009,488.00",-23.74
336,T5_Base_split_4/0012_T5_Base_multiple_2_2_2_2_...,0,6291456,"((8, 12), (9, 13), (10, 14), (11, 15))","6,476,080,262.00","8,479,339,787.00","-2,003,259,525.00",-23.63
393,T5_Base_split_4/0071_T5_Base_multiple_1_4_2_2_...,0,3145728,"((0, 4), (1, 5), (2, 6), (3, 7))","3,238,040,020.00","4,218,537,162.00","-980,497,142.00",-23.24
369,T5_Base_split_4/0047_T5_Base_multiple_4_2_2_1_...,2,8388608,"((0, 4), (1, 5), (2, 6), (3, 7), (8, 12), (9, ...","8,634,700,020.00","11,133,745,404.00","-2,499,045,384.00",-22.45
